In [1]:
   import sqlite3
   import pandas as pd

   conn = sqlite3.connect("data/sports.db")

   games = pd.read_sql("""
       SELECT g.id, g.season, g.date,
              h.name AS home, a.name AS away,
              g.home_score, g.away_score
       FROM games g
       JOIN teams h ON h.id = g.home_team_id
       JOIN teams a ON a.id = g.away_team_id
       WHERE g.status = 'final'
       ORDER BY g.date
   """, conn)

   games.tail()
   

,id,season,date,home,away,home_score,away_score
12749,12750,2026,2026-09-19,Nott'm Forest,Coventry,0,1
12750,12751,2026,2026-09-20,Bournemouth,Liverpool,0,1
12751,12752,2026,2026-09-20,Leeds,Crystal Palace,0,0
12752,12753,2026,2026-09-20,Man City,Sunderland,5,3
12753,12754,2026,2026-09-20,Fulham,Man United,1,1


In [2]:
from scipy.stats import poisson

recent = games[games["season"] >= 2021]
avg_home = recent["home_score"].mean()

actual = recent["home_score"].value_counts(normalize=True).sort_index()
comparison = pd.DataFrame({
    "actual": actual,
    "poisson": [poisson.pmf(k, avg_home) for k in actual.index],
}).round(3)

print(f"Average home goals since 2021/22: {avg_home:.2f}")
comparison.head(7)

Average home goals since 2021/22: 1.60


,actual,poisson
home_score,,
0,0.223,0.203
1,0.305,0.324
2,0.251,0.258
3,0.136,0.137
4,0.058,0.055
5,0.020,0.017
6,0.005,0.005


In [5]:
import numpy as np
from scipy.optimize import minimize

def fit_poisson(train, as_of, half_life_days=180, shrinkage=3.0):
    teams = sorted(set(train["home"]) | set(train["away"]))
    index = {team: i for i, team in enumerate(teams)}
    n = len(teams)

    home_idx = train["home"].map(index).to_numpy()
    away_idx = train["away"].map(index).to_numpy()
    home_goals = train["home_score"].to_numpy()
    away_goals = train["away_score"].to_numpy()

    days_ago = (pd.Timestamp(as_of) - pd.to_datetime(train["date"])).dt.days.to_numpy()
    weights = 0.5 ** (days_ago / half_life_days)

    def unpack(params):
        attack = params[:n] - params[:n].mean()
        defense = params[n:2 * n] - params[n:2 * n].mean()
        return attack, defense, params[2 * n], params[2 * n + 1]

    def neg_log_likelihood(params):
        attack, defense, base, home_adv = unpack(params)
        exp_home = np.exp(base + home_adv + attack[home_idx] - defense[away_idx])
        exp_away = np.exp(base + attack[away_idx] - defense[home_idx])
        log_lik = poisson.logpmf(home_goals, exp_home) + poisson.logpmf(away_goals, exp_away)
        return -np.sum(weights * log_lik) + shrinkage * np.sum(attack**2 + defense**2)

    result = minimize(neg_log_likelihood, np.zeros(2 * n + 2), method="L-BFGS-B")
    if not result.success:
        print("Warning: optimizer did not converge:", result.message)

    attack, defense, base, home_adv = unpack(result.x)
    return {
        "ratings": pd.DataFrame({"attack": np.exp(attack), "defense": np.exp(defense)}, index=teams),
        "base": np.exp(base),
        "home_adv": np.exp(home_adv),
    }

In [6]:
as_of = games["date"].max()
train = games[games["date"] >= "2023-07-01"]
model = fit_poisson(train, as_of)

print(f"Base expected goals: {model['base']:.2f}")
print(f"Home advantage: x{model['home_adv']:.2f}")

current_teams = games.loc[games["season"] == 2026, "home"].unique()
model["ratings"].loc[current_teams].sort_values("attack", ascending=False).round(2)

Base expected goals: 1.23
Home advantage: x1.19


,attack,defense
Man City,1.50,1.34
Brighton,1.30,1.13
Arsenal,1.28,1.60
Man United,1.25,1.05
Liverpool,1.24,1.12
Chelsea,1.21,0.92
Newcastle,1.19,0.94
Brentford,1.16,1.10
Bournemouth,1.09,1.07
Aston Villa,1.09,1.00


In [7]:
def predict(model, home, away, max_goals=10):
    r = model["ratings"]
    exp_home = model["base"] * model["home_adv"] * r.loc[home, "attack"] / r.loc[away, "defense"]
    exp_away = model["base"] * r.loc[away, "attack"] / r.loc[home, "defense"]

    goals = np.arange(max_goals + 1)
    grid = np.outer(poisson.pmf(goals, exp_home), poisson.pmf(goals, exp_away))

    return {
        "exp_home": exp_home,
        "exp_away": exp_away,
        "home": np.tril(grid, -1).sum(),
        "draw": np.trace(grid),
        "away": np.triu(grid, 1).sum(),
        "grid": grid,
    }

In [8]:
p = predict(model, "Arsenal", "Man City")

print(f"Expected goals: Arsenal {p['exp_home']:.2f} - {p['exp_away']:.2f} Man City")
print(f"Home win {p['home']:.1%}   Draw {p['draw']:.1%}   Away win {p['away']:.1%}")
print(f"Fair odds: {1/p['home']:.2f} / {1/p['draw']:.2f} / {1/p['away']:.2f}")

pd.DataFrame(
    p["grid"][:5, :5] * 100,
    index=pd.Index(range(5), name="Arsenal goals"),
    columns=pd.Index(range(5), name="Man City goals"),
).round(1)

Expected goals: Arsenal 1.41 - 1.16 Man City
Home win 42.7%   Draw 26.4%   Away win 30.9%
Fair odds: 2.34 / 3.79 / 3.23


Man City goals,0,1,2,3,4
Arsenal goals,,,,,
0,7.7,8.9,5.2,2.0,0.6
1,10.8,12.5,7.3,2.8,0.8
2,7.6,8.8,5.1,2.0,0.6
3,3.6,4.1,2.4,0.9,0.3
4,1.3,1.5,0.8,0.3,0.1


In [9]:
market = pd.read_sql("""
    WITH closing AS (
        SELECT game_id, bookmaker,
               MAX(CASE WHEN outcome = 'home' THEN 1.0 / price END) AS h,
               MAX(CASE WHEN outcome = 'draw' THEN 1.0 / price END) AS d,
               MAX(CASE WHEN outcome = 'away' THEN 1.0 / price END) AS a
        FROM odds
        WHERE market = '1x2' AND is_closing = 1
          AND bookmaker IN ('Pinnacle', 'Betfair Exchange')
        GROUP BY game_id, bookmaker
        HAVING COUNT(*) = 3
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
                      PARTITION BY game_id
                      ORDER BY bookmaker = 'Pinnacle' DESC) AS rn
        FROM closing
    )
    SELECT game_id AS id, bookmaker AS benchmark,
           h / (h + d + a) AS mkt_home,
           d / (h + d + a) AS mkt_draw,
           a / (h + d + a) AS mkt_away
    FROM ranked
    WHERE rn = 1
""", conn)

market["benchmark"].value_counts()

benchmark
Pinnacle            5150
Betfair Exchange     198
Name: count, dtype: int64

In [10]:
def backtest(games, start_season, **model_settings):
    test = games[games["season"] >= start_season].copy()
    test["week"] = pd.to_datetime(test["date"]).dt.to_period("W").dt.start_time

    rows = []
    for week_start, week_games in test.groupby("week"):
        cutoff = week_start.strftime("%Y-%m-%d")
        window_start = (week_start - pd.DateOffset(years=3)).strftime("%Y-%m-%d")
        train = games[(games["date"] >= window_start) & (games["date"] < cutoff)]
        model = fit_poisson(train, cutoff, **model_settings)
        known = set(model["ratings"].index)

        for g in week_games.itertuples():
            if g.home not in known or g.away not in known:
                continue  # a team with no games in the training window yet
            p = predict(model, g.home, g.away)
            if g.home_score > g.away_score:
                result = "home"
            elif g.home_score == g.away_score:
                result = "draw"
            else:
                result = "away"
            rows.append({"id": g.id, "season": g.season, "result": result,
                         "model_home": p["home"], "model_draw": p["draw"], "model_away": p["away"]})
    return pd.DataFrame(rows)

results = backtest(games, start_season=2016)
len(results)

3833

In [11]:
def add_scores(df, prefix):
    outcomes = ["home", "draw", "away"]
    probs = df[[f"{prefix}_{o}" for o in outcomes]].to_numpy()
    actual = (df["result"].to_numpy()[:, None] == np.array(outcomes)).astype(float)
    df[f"{prefix}_logloss"] = -np.log((probs * actual).sum(axis=1))
    df[f"{prefix}_brier"] = ((probs - actual) ** 2).sum(axis=1)

scored = results.merge(market, on="id")
add_scores(scored, "model")
add_scores(scored, "mkt")

summary = scored.groupby("season")[["model_logloss", "mkt_logloss", "model_brier", "mkt_brier"]].mean()
summary.insert(0, "games", scored.groupby("season").size())
print(scored[["model_logloss", "mkt_logloss", "model_brier", "mkt_brier"]].mean().round(4))
summary.round(4)

model_logloss    0.9705
mkt_logloss      0.9507
model_brier      0.5762
mkt_brier        0.5628
dtype: float64


,games,model_logloss,mkt_logloss,model_brier,mkt_brier
season,,,,,
2016,379,0.9427,0.9056,0.5568,0.5317
2017,378,0.9589,0.9402,0.5693,0.5568
2018,377,0.8949,0.8900,0.5234,0.5203
2019,377,0.9780,0.9750,0.5815,0.5763
2020,379,1.0170,0.9990,0.6063,0.5935
2021,379,0.9593,0.9355,0.5682,0.5535
2022,379,0.9870,0.9634,0.5882,0.5721
2023,379,0.9425,0.9012,0.5557,0.5270
2024,379,0.9841,0.9681,0.5876,0.5763
